# Archivo Python | Proyecto Integrador RavenStack
## Autora: Marta Quevedo Oltra | Unicorn Edition 13.0
_______________________________________________________________
### Este archivo consta de tres bloques:
#### - BLOQUE 1 : Descarga del dataset original desde Kaggle
#### - BLOQUE 2: Análisis Exploratorio de Datos (EDA)
#### - BLOQUE 3: Ingesta de datos a MySQL

_______________________________________________________________

## Instalación de dependencias 

Se instalan cuatro dependencias:

- **kagglehub** descarga el dataset original directamente desde Kaggle mediante su API.
- **python-dotenv** lee las credenciales de MySQL desde un archivo `.env` externo, de modo que no figuren en el código.
- **SQLAlchemy** traduce las operaciones de pandas a SQL y gestiona la conexión con la base de datos.
- **PyMySQL** es el driver que implementa el protocolo de comunicación con MySQL.

La combinación de las dos últimas permite escribir los DataFrames directamente en las tablas mediante `df.to_sql()`, cerrando el flujo Kaggle API → Python → MySQL de forma reproducible y sin cargas manuales. 

In [1]:
!pip install kagglehub python-dotenv sqlalchemy pymysql


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importación de librerías

In [ ]:
# --- Manipulación de datos 
import pandas as pd
import numpy as np

# --- Sistema y configuración ---
import os
from dotenv import load_dotenv, find_dotenv
from urllib.parse import quote_plus

# --- Extracción y carga ---
import kagglehub
from sqlalchemy import create_engine, text

___________________________________________________________________

## BLOQUE 1. Descarga del dataset desde Kaggle

In [3]:
# Download latest version
path = kagglehub.dataset_download("rivalytics/saas-subscription-and-churn-analytics-dataset")

print("Archivos disponibles:", os.listdir(path))

Archivos disponibles: ['ravenstack_accounts.csv', 'ravenstack_churn_events.csv', 'ravenstack_feature_usage.csv', 'ravenstack_subscriptions.csv', 'ravenstack_support_tickets.csv', 'README.md']


#### Carga de las 5 tablas del dataset extraído desde Kaggle

In [4]:
accounts = pd.read_csv(os.path.join(path, "ravenstack_accounts.csv"))
subs = pd.read_csv(os.path.join(path, "ravenstack_subscriptions.csv"))
usage = pd.read_csv(os.path.join(path, "ravenstack_feature_usage.csv"))
churn = pd.read_csv(os.path.join(path, "ravenstack_churn_events.csv"))
tickets = pd.read_csv(os.path.join(path, "ravenstack_support_tickets.csv"))

___________________________________________________________________

## BLOQUE 2. Análisis Exploratorio de Datos (EDA) 

Este bloque se ejecuta desde el dataset original de Kaggle

### Accounts (tabla de dimensiones)

In [5]:
accounts.info() 

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   account_id       500 non-null    str  
 1   account_name     500 non-null    str  
 2   industry         500 non-null    str  
 3   country          500 non-null    str  
 4   signup_date      500 non-null    str  
 5   referral_source  500 non-null    str  
 6   plan_tier        500 non-null    str  
 7   seats            500 non-null    int64
 8   is_trial         500 non-null    bool 
 9   churn_flag       500 non-null    bool 
dtypes: bool(2), int64(1), str(7)
memory usage: 32.4 KB


La columna 'signup_date' viene en formato texto (string) y eso significa que se va a tener que parsear las fechas para poder realizar cualquier cálculo temporal. 

#### Parseado de fechas de la columna signup_date:

In [6]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])

In [7]:
accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   account_id       500 non-null    str           
 1   account_name     500 non-null    str           
 2   industry         500 non-null    str           
 3   country          500 non-null    str           
 4   signup_date      500 non-null    datetime64[us]
 5   referral_source  500 non-null    str           
 6   plan_tier        500 non-null    str           
 7   seats            500 non-null    int64         
 8   is_trial         500 non-null    bool          
 9   churn_flag       500 non-null    bool          
dtypes: bool(2), datetime64[us](1), int64(1), str(6)
memory usage: 32.4 KB


#### Comprobación de métricas principales: 

In [8]:
accounts.describe(include = 'str')

,account_id,account_name,industry,country,referral_source,plan_tier
count,500,500,500,500,500,500
unique,500,500,5,7,5,3
top,A-2e4581,Company_0,DevTools,US,organic,Pro
freq,1,1,113,291,114,178


Esta función nos ayuda a entender un poco mejor la tabla "Accounts".

Leyendo la métrica 'count' y 'unique' sobre 'account_id', ya podemos comprobar que no hay duplicados, porque los registros coinciden. 


Leyendo la métrica top y freq (que van juntas) podemos leer ya varios puntos:

- **DevTools** es la industria que más veces se repite.
- **Estados Unidos** es el país dominante.
- El **plan de subscripción 'Pro'** es el plan más contratado con 178 cuentas. 

In [9]:
accounts.describe(include = 'int64')

,seats
count,500.000000
mean,20.560000
std,21.044718
min,1.000000
25%,5.000000
50%,15.000000
75%,28.000000
max,163.000000


In [10]:
accounts.describe(include = 'datetime')

,signup_date
count,500
mean,2024-01-27 06:34:33.600000
min,2023-01-02 00:00:00
25%,2023-08-06 18:00:00
50%,2024-02-21 00:00:00
75%,2024-08-04 06:00:00
max,2024-12-31 00:00:00


#### Comprobación de nulos 

In [11]:
accounts.isnull().sum()

account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

Esta tabla viene sin nulos.

In [12]:
accounts.describe(include = 'bool')

,is_trial,churn_flag
count,500,500
unique,2,2
top,False,False
freq,403,390


### Subscriptions (tabla de dimensiones o híbrida)

In [13]:
subs.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   subscription_id    5000 non-null   str  
 1   account_id         5000 non-null   str  
 2   start_date         5000 non-null   str  
 3   end_date           486 non-null    str  
 4   plan_tier          5000 non-null   str  
 5   seats              5000 non-null   int64
 6   mrr_amount         5000 non-null   int64
 7   arr_amount         5000 non-null   int64
 8   is_trial           5000 non-null   bool 
 9   upgrade_flag       5000 non-null   bool 
 10  downgrade_flag     5000 non-null   bool 
 11  churn_flag         5000 non-null   bool 
 12  billing_frequency  5000 non-null   str  
 13  auto_renew_flag    5000 non-null   bool 
dtypes: bool(5), int64(3), str(6)
memory usage: 376.1 KB


- Mismo problema de tratamiento de fechas (se necesita parsear). 

- Se detectan 4514 nulos en la columna end_date, pero se trata de nulos que tienen razón de ser ya que significa que de 5000 filas de subscripción, 4514 están activas (no tienen fecha de finalización). 

#### Parseado de fechas de la columna start_date y signup_date:

In [14]:
subs['start_date'] = pd.to_datetime(subs['start_date'])
subs['end_date'] = pd.to_datetime(subs['end_date'])

In [15]:
subs.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   subscription_id    5000 non-null   str           
 1   account_id         5000 non-null   str           
 2   start_date         5000 non-null   datetime64[us]
 3   end_date           486 non-null    datetime64[us]
 4   plan_tier          5000 non-null   str           
 5   seats              5000 non-null   int64         
 6   mrr_amount         5000 non-null   int64         
 7   arr_amount         5000 non-null   int64         
 8   is_trial           5000 non-null   bool          
 9   upgrade_flag       5000 non-null   bool          
 10  downgrade_flag     5000 non-null   bool          
 11  churn_flag         5000 non-null   bool          
 12  billing_frequency  5000 non-null   str           
 13  auto_renew_flag    5000 non-null   bool          
dtypes: bool(5), datetim

In [16]:
subs['end_date'].isnull().sum()

np.int64(4514)

De 5000 registros, 4514 nulos en la columna 'end_date' nos dicen que esos nulos son subscripciones activas. 
Por lo que hay 486 subscripciones que tienen fecha de cancelación.

#### Comprobación de métricas principales: 

In [17]:
subs.describe()

,start_date,end_date,seats,mrr_amount,arr_amount
count,5000,486,5000.000000,5000.000000,5000.000000
mean,2024-07-14 01:49:43.680000,2024-09-29 09:14:04.444444,29.852000,2267.749400,27212.992800
min,2023-01-09 00:00:00,2023-04-05 00:00:00,1.000000,0.000000,0.000000
25%,2024-04-17 18:00:00,2024-08-19 12:00:00,14.000000,285.000000,3420.000000
50%,2024-09-01 00:00:00,2024-11-10 12:00:00,24.000000,931.000000,11172.000000
75%,2024-11-17 00:00:00,2024-12-18 18:00:00,40.000000,2786.000000,33432.000000
max,2024-12-31 00:00:00,2024-12-31 00:00:00,189.000000,33830.000000,405960.000000
std,NaN,NaN,23.089771,3421.375348,41056.504178


In [18]:
subs.describe(include = 'str')

,subscription_id,account_id,plan_tier,billing_frequency
count,5000,5000,5000,5000
unique,5000,500,3,2
top,S-8cec59,A-d4ac0e,Enterprise,monthly
freq,1,19,1723,2539


In [19]:
subs.describe(include = 'bool')

,is_trial,upgrade_flag,downgrade_flag,churn_flag,auto_renew_flag
count,5000,5000,5000,5000,5000
unique,2,2,2,2,2
top,False,False,False,False,True
freq,4222,4471,4782,4514,4005


### Feature_usage (tabla de hechos)

In [20]:
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   usage_id             25000 non-null  str  
 1   subscription_id      25000 non-null  str  
 2   usage_date           25000 non-null  str  
 3   feature_name         25000 non-null  str  
 4   usage_count          25000 non-null  int64
 5   usage_duration_secs  25000 non-null  int64
 6   error_count          25000 non-null  int64
 7   is_beta_feature      25000 non-null  bool 
dtypes: bool(1), int64(3), str(4)
memory usage: 1.4 MB


Mismo problema de tratamiento de fechas (se necesita parsear). 

#### Parseado de fechas de la columna usage_date

In [21]:
usage['usage_date'] = pd.to_datetime(usage['usage_date'])

In [22]:
usage.info() #comprobación

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   usage_id             25000 non-null  str           
 1   subscription_id      25000 non-null  str           
 2   usage_date           25000 non-null  datetime64[us]
 3   feature_name         25000 non-null  str           
 4   usage_count          25000 non-null  int64         
 5   usage_duration_secs  25000 non-null  int64         
 6   error_count          25000 non-null  int64         
 7   is_beta_feature      25000 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(3), str(3)
memory usage: 1.4 MB


In [23]:
usage.describe(include = 'datetime')

,usage_date
count,25000
mean,2023-12-31 21:23:47.328000
min,2023-01-01 00:00:00
25%,2023-06-30 00:00:00
50%,2024-01-02 00:00:00
75%,2024-07-03 00:00:00
max,2024-12-31 00:00:00


**Se registra que el período de uso va desde enero de 2023 hasta diciembre de 2024 (2 años).**

In [24]:
usage.describe(include = 'str')

,usage_id,subscription_id,feature_name
count,25000,25000,25000
unique,24979,4967,40
top,U-25b56c,S-0896f4,feature_12
freq,2,16,659


#### Hallazgo: 
Hay 4967 subscription_id únicos en esta tabla. Teniendo en cuenta que hay 5000 registros de suscripciones que figuran en la tabla 'suscripcions', **faltan 33 suscripciones las cuales no tienen registrado ningún uso (¿posibles indicadores tempranos de churn?)**

#### Hallazgo: duplicados en usage_id
21 IDs duplicados (42 filas) en feature_usage. Verificado que las filas son eventos distintos (diferente subscription_id, fecha y feature), no duplicación de datos.

In [25]:
# Comprobación de duplicados:
usage[usage['usage_id'].duplicated(keep=False)].sort_values('usage_id')

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


Como se trata de duplicados solamente en el usage_id pero los registros parecen ser diferentes, **se decide mantener los datos y no borrar esos duplicados**, ya que puede deberse a un error de generación del id en usage. 

In [26]:
usage.describe(include = 'datetime64')

,usage_date
count,25000
mean,2023-12-31 21:23:47.328000
min,2023-01-01 00:00:00
25%,2023-06-30 00:00:00
50%,2024-01-02 00:00:00
75%,2024-07-03 00:00:00
max,2024-12-31 00:00:00


In [27]:
usage.describe()

,usage_date,usage_count,usage_duration_secs,error_count
count,25000,25000.000000,25000.000000,25000.000000
mean,2023-12-31 21:23:47.328000,10.021000,3042.202880,0.564280
min,2023-01-01 00:00:00,0.000000,0.000000,0.000000
25%,2023-06-30 00:00:00,8.000000,1350.000000,0.000000
50%,2024-01-02 00:00:00,10.000000,2760.000000,0.000000
75%,2024-07-03 00:00:00,12.000000,4400.000000,1.000000
max,2024-12-31 00:00:00,26.000000,12696.000000,8.000000
std,NaN,3.143729,2056.544615,1.012595


### Churn_events (tabla de hechos)

In [28]:
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    str    
 1   account_id                600 non-null    str    
 2   churn_date                600 non-null    str    
 3   reason_code               600 non-null    str    
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    str    
dtypes: bool(3), float64(1), str(5)
memory usage: 30.0 KB


Mismo problema de fechas. Se parsean:

In [29]:
churn['churn_date'] = pd.to_datetime(churn['churn_date'])

In [30]:
churn.info() 

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   churn_event_id            600 non-null    str           
 1   account_id                600 non-null    str           
 2   churn_date                600 non-null    datetime64[us]
 3   reason_code               600 non-null    str           
 4   refund_amount_usd         600 non-null    float64       
 5   preceding_upgrade_flag    600 non-null    bool          
 6   preceding_downgrade_flag  600 non-null    bool          
 7   is_reactivation           600 non-null    bool          
 8   feedback_text             452 non-null    str           
dtypes: bool(3), datetime64[us](1), float64(1), str(4)
memory usage: 30.0 KB


In [31]:
churn.describe(include = 'str')

,churn_event_id,account_id,reason_code,feedback_text
count,600,600,600,452
unique,600,352,6,3
top,C-816288,A-0a62f5,features,too expensive
freq,1,5,114,161


Me encuentro con que tanto en la fuente de origen como los datos aquí cargados, en esta exploración de tablas cuentan historias muy distintas Y CONTRADICTORIAS sobre el porcentaje de churn:

- #### En la tabla churn_events, columna 'accound_id': 
Hay 352 cuentas únicas con evento de churn. Eso significa un 70,4% de 500 cuentas.
  
- #### En la tabla accounts, columna 'churn_flag':
En cambio ahí se registran 390 eventos de churn FALSOS, lo cual eso es un 78% de cuentas activas frente 22% de churn.

**Dado que en este punto los datos pueden parecer confusos, así que me propongo entender:** ¿Cuántas cuentas con 'churn_flag = True' tienen evento en la tabla de churn_events?

In [50]:
cuentas_churn_events = set(churn['account_id'].unique())  #cuentas únicas en la tabla de churn_events
cuentas_flag_true = set(accounts[accounts['churn_flag'] == True]['account_id']) #cuentas que tienen un True en'churn_flag'

**Las unimos para entender los números:**

In [33]:
print("Cuentas registradas en la tabla de churn_events pero tienen 'False' en churn_flag:", len(cuentas_churn_events - cuentas_flag_true)) 
print("Cuentas sin registro en la tabla de churn_events pero tienen 'True' en churn_flag:", len(cuentas_flag_true - cuentas_churn_events))
print("Coinciden en ambos registros:", len(cuentas_churn_events & cuentas_flag_true))

Cuentas registradas en la tabla de churn_events pero tienen 'False' en churn_flag: 277
Cuentas sin registro en la tabla de churn_events pero tienen 'True' en churn_flag: 35
Coinciden en ambos registros: 75


#### **Hallazgo**:
'Churn_flag' en la tabla accounts tiene 35 registros 'True', pero no figuran en la tabla de churn_events.
En cambio, sí hay 277 cuentas registradas como canceladas en la tabla de churn_events, que en cambio, en la tabla accounts cuentan como activas: churn_flag = 'False'. 

Solo coinciden en ambos registros, tanto registradas en eventos de churn como en la tabla de accounts como canceladas: 75 cuentas.

**No hay manera de reconciliar estos números, y no se entiende que una cuenta que figura como cancelada no tenga registro en la tabla de churn_events (35 cuentas).**

El dataset en este sentido es internamente inconsistente así que hay que tomar una decisión --> **Elegir una única fuente de verdad.**

#### **Decisiones**:

- **Logo churn** = 352 cuentas únicas registradas en churn_events.
- **Porcentaje** = 352/500*100 = 70,4% de cuentas que han churneado alguna vez durante el período de dos años: 2023-2024.
- **La columna 'churn_flag' de la tabla accounts queda descartada como métrica de churn**.

In [34]:
unicos = churn['reason_code'].unique()
unicos

<StringArray>
['pricing', 'support', 'budget', 'unknown', 'features', 'competitor']
Length: 6, dtype: str

In [35]:
unicos = churn['feedback_text'].unique()
unicos

<StringArray>
['switched to competitor', nan, 'missing features', 'too expensive']
Length: 4, dtype: str

In [36]:
churn.describe(include = 'float')

,refund_amount_usd
count,600.000000
mean,14.420417
std,39.224591
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,392.920000


Más del 75% de los eventos de churn tuvieron reembolso cero. La mayoría de cancelaciones no generan reembolso; solo una minoría sí, y algunas son elevadas. 

El reembolso más alto registrado en un evento es de 392.92 USD.

In [37]:
churn.describe(include = 'bool')

,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation
count,600,600,600
unique,2,2,2
top,False,False,False
freq,477,547,539


#### **Hallazgos**:
- 123 eventos de churn presentaron una subida de nivel de la suscripción churneada los 90 días previos a la fecha de cancelación. (**20,5% de casos de *upgrade* previo al churn**).
  
- 53 cuentas presentaron una degradación del nivel de la suscripción churneada los 90 días previos a la fecha de cancelación. (**8,8% de casos de *downgrade* previo al churn**).

  
- De 600 cancelaciones, **61 suscripciones han sido reactivadas**.

### Support_tickets (tabla de hechos)

In [38]:
tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    2000 non-null   str    
 1   account_id                   2000 non-null   str    
 2   submitted_at                 2000 non-null   str    
 3   closed_at                    2000 non-null   str    
 4   resolution_time_hours        2000 non-null   float64
 5   priority                     2000 non-null   str    
 6   first_response_time_minutes  2000 non-null   int64  
 7   satisfaction_score           1175 non-null   float64
 8   escalation_flag              2000 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(5)
memory usage: 127.1 KB


Mismo problema de formato de fechas. Se parsean las columnas 'submitted_at' y 'closed_at':

In [39]:
tickets['submitted_at'] = pd.to_datetime(tickets['submitted_at'])
tickets['closed_at'] = pd.to_datetime(tickets['closed_at'])

In [40]:
tickets.info() #comprobamos

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ticket_id                    2000 non-null   str           
 1   account_id                   2000 non-null   str           
 2   submitted_at                 2000 non-null   datetime64[us]
 3   closed_at                    2000 non-null   datetime64[us]
 4   resolution_time_hours        2000 non-null   float64       
 5   priority                     2000 non-null   str           
 6   first_response_time_minutes  2000 non-null   int64         
 7   satisfaction_score           1175 non-null   float64       
 8   escalation_flag              2000 non-null   bool          
dtypes: bool(1), datetime64[us](2), float64(2), int64(1), str(3)
memory usage: 127.1 KB


In [41]:
tickets.describe(include = 'str')

,ticket_id,account_id,priority
count,2000,2000,2000
unique,2000,492,4
top,T-0024de,A-bb3bd4,urgent
freq,1,11,514


In [42]:
tickets.describe()

,submitted_at,closed_at,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,2000,2000,2000.000000,2000.000000,1175.000000
mean,2024-01-05 14:08:09.600000,2024-01-07 01:59:49.200000,35.861000,88.480000,3.981277
min,2023-01-02 00:00:00,2023-01-03 03:00:00,1.000000,1.000000,3.000000
25%,2023-07-09 18:00:00,2023-07-10 21:00:00,17.000000,43.000000,3.000000
50%,2024-01-06 12:00:00,2024-01-08 05:00:00,35.000000,88.000000,4.000000
75%,2024-07-09 06:00:00,2024-07-10 11:00:00,54.000000,131.000000,5.000000
max,2024-12-31 00:00:00,2024-12-31 19:00:00,72.000000,180.000000,5.000000
std,NaN,NaN,21.138427,51.531877,0.809646


In [43]:
tickets.describe(include = 'int')

,first_response_time_minutes
count,2000.000000
mean,88.480000
std,51.531877
min,1.000000
25%,43.000000
50%,88.000000
75%,131.000000
max,180.000000


In [44]:
tickets.describe(include = 'bool')

,escalation_flag
count,2000
unique,2
top,False
freq,1905


**Observación de fondo completa**: 
- Tiempo de resolución máxima en 72h.
- Tiempo de respuesta máxima 3h.
- Satisfacción mínima: 3.
- Solo el 4,75% de los tickets han escalado.

El soporte de RavenStack parece funcionar razonablemente bien en todas las métricas. 

Comprobación de valores para la creación de tablas en MySQL (usando la función ENUM): 

In [45]:
accounts['industry'].unique()

<StringArray>
['EdTech', 'FinTech', 'DevTools', 'HealthTech', 'Cybersecurity']
Length: 5, dtype: str

In [46]:
accounts['referral_source'].unique()

<StringArray>
['partner', 'other', 'organic', 'event', 'ads']
Length: 5, dtype: str

In [47]:
accounts['plan_tier'].unique()

<StringArray>
['Basic', 'Enterprise', 'Pro']
Length: 3, dtype: str

___________________________________________________________________

## BLOQUE 3 Ingesta de datos a MySQL

Este bloque se ejecuta a partir del **dataset gemelo**:

**ORDEN OBLIGATORIO**:

- A) **01_ddl_vistas_ravenstack_gemelo.sql**    -> crea la base y las tablas VACIAS a partir del **Archivo DDL**.

- B) **01_ravenstack_extraccion_EDA_ingesta**   -> carga los datos en este **BLOQUE 3**.

- C) **01_ddl_vistas_ravenstack_gemelo**        -> volver al archivo de SQL y ejecutar el código de **Vistas Base para el Análisis de Churn**.

Las vistas van DESPUÉS de la ingesta de datos: se pueden crear sobre tablas vacias, pero sus consultas de verificacion devolverian cero y parecerian rotas.

#### **1. CONFIGURACION**  ---  las credenciales NO viven en este fichero

Copiar el documento adjunto .env.example como .env y rellenar las credenciales de MySQL y las rutas a los datos. 

In [ ]:
load_dotenv(find_dotenv(), override=True)

# Lectura de las variables definidas en el .env:
USUARIO    = os.getenv("DB_USER")
CONTRASENA = os.getenv("DB_PASSWORD")
HOST       = os.getenv("DB_HOST", "localhost")
PUERTO     = os.getenv("DB_PORT", "3306")
BASE       = os.getenv("DB_NAME_GEMELO", "ravenstack_gemelo")
CARPETA    = os.getenv("RUTA_DATOS_GEMELO")

# Comprobación de que la configuración se ha cargado correctamente
print("Base de datos:", BASE)
print("Carpeta de datos:", CARPETA)
print("¿Existe la carpeta?", os.path.exists(CARPETA) if CARPETA else False)

In [49]:
engine = create_engine(
    f"mysql+pymysql://{USUARIO}:{quote_plus(CONTRASENA)}@{HOST}:{PUERTO}/{BASE}")

#### **2. LECTURA DE LOS CINCO FICHEROS**

El orden de esta lista NO es decorativo: es el orden de dependencia de las claves foraneas. subscriptions referencia a accounts, y feature_usage referencia a subscriptions. Cargar en otro orden provoca un error de integridad referencial.

In [ ]:
TABLAS = [
    ("accounts",        "ravenstack_accounts.csv"),
    ("subscriptions",   "ravenstack_subscriptions.csv"),
    ("feature_usage",   "ravenstack_feature_usage.csv"),
    ("churn_events",    "ravenstack_churn_events.csv"),
    ("support_tickets", "ravenstack_support_tickets.csv"),
]

In [ ]:
datos = {}
for tabla, fichero in TABLAS:
    ruta = os.path.join(CARPETA, fichero)
    datos[tabla] = pd.read_csv(ruta)
    print(f"{tabla:<17} {datos[tabla].shape[0]:>6} filas  {datos[tabla].shape[1]:>2} columnas")

#### **3. CARGA**

if_exists='append' es OBLIGATORIO.

Con 'replace', pandas ELIMINA la tabla y la vuelve a crear con un esquema inferido por el: se pierden los ENUM, los NOT NULL, las claves primarias y
las cinco claves foraneas del DDL. La base quedaria sin ninguna restriccionde integridad y el trabajo de la Fase 2 desaparecido.

index=False evita que pandas anada su indice como columna extra.

In [ ]:
for tabla, _ in TABLAS:
    datos[tabla].to_sql(tabla, con=engine, if_exists="append", index=False)
    print(f"cargada -> {tabla}")

#### **4. VERIFICACION DE LA CARGA**

In [ ]:
ESPERADO = {
    "accounts": 500,
    "subscriptions": 1164,
    "feature_usage": 25000,
    "churn_events": 238,
    "support_tickets": 2000,
}

In [ ]:
print("\n--- RECUENTO DE FILAS ---")
with engine.connect() as con:
    for tabla, esperado in ESPERADO.items():
        n = con.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
        print(f"  {'OK ' if n == esperado else 'MAL'} {tabla:<17}{n:>7} (esperado {esperado})")

___________________________________________________________________